In [1]:
import re
import pandas as pd
import tqdm

tqdm.tqdm.pandas()
def func(s):
    if not isinstance(s, str):
        return []  # Return empty list for non-string inputs
    
    dict_pattern = r'\{[^}]+\}'
    dict_matches = re.findall(dict_pattern, s)
    kv_pattern = r"'([^']+)':\s([^,}]+)"
    extracted_data = []

    for dict_str in dict_matches:  # Limit to first 10 dictionaries
        kv_matches = re.findall(kv_pattern, dict_str)
        extracted_dict = {key: value.strip() for key, value in kv_matches}
        extracted_data.append(extracted_dict)
    
    return extracted_data
result = []
# Read the CSV in chunks
chunk_size = 5000  # Adjust this based on your available memory
for chunk in pd.read_csv(r"merged_df.csv", chunksize=chunk_size):
    chunk["prev_72h_weather"] = chunk['prev_72h_weather'].progress_apply(func)
    result.append(chunk)
    # Process or save the chunk here
    print("Processed chunk")
merged_df = pd.concat(result, ignore_index=True)
print("All chunks processed")


100%|██████████| 5000/5000 [00:04<00:00, 1007.82it/s]


Processed chunk


100%|██████████| 5000/5000 [00:04<00:00, 1022.71it/s]


Processed chunk


100%|██████████| 5000/5000 [00:05<00:00, 908.21it/s] 


...

Processed chunk


100%|██████████| 5000/5000 [00:11<00:00, 451.62it/s]


Processed chunk


100%|██████████| 137/137 [00:00<00:00, 641.36it/s]


Processed chunk
All chunks processed


In [12]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35137 entries, 0 to 35136
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              35137 non-null  object 
 1   begin_date_time                         35137 non-null  object 
 2   cz_timezone                             35137 non-null  object 
 3   end_date_time                           35137 non-null  object 
 4   begin_lat                               35137 non-null  float64
 5   begin_lon                               35137 non-null  float64
 6   end_lat                                 35137 non-null  float64
 7   end_lon                                 35137 non-null  float64
 8   extreme                                 35137 non-null  int64  
 9   begin_date_utc                          35137 non-null  object 
 10  begin_time_utc                          35137 non-null  ob

In [4]:
import ast
def conver_to_float(s):
    try:
        return [float(i) for i in ast.literal_eval(s)]
    except Exception as e:
        print(e)

        return None
    
merged_df_copy = merged_df.copy()
prev_cols = [col for col in merged_df_copy.columns if 'prev_72h' in col]
prev_cols.remove('prev_72h_weather')
for col in prev_cols:
    merged_df_copy[col] = merged_df_copy[col].progress_apply(conver_to_float)


100%|██████████| 35137/35137 [00:06<00:00, 5304.43it/s]


In [ ]:
merged_df_copy

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_direction_100m,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,...,"[75.5559310913086, 77.90525817871094, 78.54128...","[30.96000099182129, 33.47999954223633, 34.2000...","[21.88249969482422, 21.932498931884766, 21.482...","[19.182498931884766, 19.38249969482422, 19.482...","[19.182498931884766, 19.182498931884766, 19.18...","[21.682498931884766, 21.682498931884766, 21.68...","[0.33799999952316284, 0.3370000123977661, 0.33...","[0.3540000021457672, 0.3529999852180481, 0.352...","[0.3330000042915344, 0.3319999873638153, 0.331...","[0.3449999988079071, 0.3449999988079071, 0.344..."
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,...,"[198.64962768554688, 193.87754821777344, 191.1...","[39.959999084472656, 40.68000030517578, 40.319...","[24.58249855041504, 24.38249969482422, 23.8824...","[19.58249855041504, 19.782499313354492, 19.932...","[14.982500076293945, 14.982500076293945, 15.03...","[9.782500267028809, 9.782500267028809, 9.78250...","[0.3240000009536743, 0.3230000138282776, 0.321...","[0.3479999899864197, 0.3479999899864197, 0.347...","[0.37299999594688416, 0.37299999594688416, 0.3...","[0.414000004529953, 0.414000004529953, 0.41400..."
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,...,"[55.645606994628906, 56.97612762451172, 57.002...","[47.15999984741211, 45.36000061035156, 44.6399...","[21.032499313354492, 21.782499313354492, 22.23...","[20.932498931884766, 21.032499313354492, 21.13...","[22.732500076293945, 22.732500076293945, 22.73...","[23.08249855041504, 23.08249855041504, 23.0824...","[0.38999998569488525, 0.3889999985694885, 0.38...","[0.4020000100135803, 0.4009999930858612, 0.400...","[0.3880000114440918, 0.3880000114440918, 0.388...","[0.37700000405311584, 0.37700000405311584, 0.3..."
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,...,"[138.81417846679688, 270.0, 311.98712158203125...","[6.479999542236328, 5.399999618530273, 7.19999...","[19.298999786376953, 19.39900016784668, 19.249...","[21.39900016784668, 21.3489990234375, 21.24900...","[25.3489990234375, 25.3489990234375, 25.298999...","[26.198999404907227, 26.198999404907227, 26.19...","[0.3709999918937683, 0.36899998784065247, 0.36...","[0.3070000112056732, 0.3070000112056732, 0.307...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.15700000524520874, 0.15700000524520874, 0.1..."
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"[173.4803009033203, 169.69520568847656, 168.69...","[26.639999389648438, 24.119998931884766, 19.79...","[30.48900032043457, 28.388999938964844, 25.088...","[19.68899917602539, 19.93899917602539, 20.0889...","[19.48900032043457, 19.48900032043457, 19.4890...","[18.538999557495117, 18.538999557495117, 18.53...","[0.04800000041723251, 0.04699999839067459, 0.0...","[0.16099999845027924, 0.16099999845027924, 0.1...","[0.22300000488758087, 0.22300000488758087, 0.2...","[0.2590000033378601, 0.2590000033378601, 0.259..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35132,Flood,2012-09-09 23:30:00,MST-7,2012-09-13 08:30:00,31.96,-112.20,31.98,-112.36,0,2012-09-10,...,"[173.15731811523438, 167.90525817871094, 176.3...","[10.079999923706055, 9.359999656677246, 10.079...","[27.671001434326172, 27.121000289916992, 26.67...","[32.72100067138672, 32.520999908447266, 32.320...","[32.42100143432617, 32.421001434326

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Masking, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ===========================
# Step 1. Data Preparation
# ===========================
df  = merged_df_copy.copy()
extreme = df[df['extreme'] == 1]
non_extreme = df[df['extreme'] == 0].sample(n=extreme.shape[0], random_state=42)
df = pd.concat([extreme, non_extreme], ignore_index=True)
# Assume df is your DataFrame.
# Identify all columns that start with "prev_72h"
prev_cols = [col for col in df.columns if col.startswith("prev_72h")]
prev_cols.remove('prev_72h_weather')
prev_cols.remove('prev_72h_snow_depth')
# We need to convert each such column (which is a list of 72 floats) into a 2D array:
# We'll create a dictionary to hold the scaled arrays.
scaled_channels = {}
scalers = {}  # To keep track of scalers for each channel if needed later

for col in prev_cols:
    # Create a numpy array of shape (n_samples, 72) from the list values
    data = np.stack(df[col].values)  # shape: (n_samples, 72)
    
    # Scale each column (time step) independently or treat the entire time series as one feature.
    # Here, we treat each time series as one feature vector and scale over the sample dimension.
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)  # scales each time step feature using samples
    
    scaled_channels[col] = data_scaled  # shape remains (n_samples, 72)
    scalers[col] = scaler

# Now combine all scaled channels along the last axis.
# We want an input array X with shape (n_samples, 72, num_channels)
# First, list the arrays in the order of prev_cols.
channel_list = [scaled_channels[col] for col in prev_cols]  # each shape: (n_samples, 72)
# Stack along the last axis: first, add a new axis for channel dimension.
channel_list = [arr[..., np.newaxis] for arr in channel_list]  # each becomes (n_samples, 72, 1)
# Concatenate along the last axis
X_seq = np.concatenate(channel_list, axis=-1)  # shape: (n_samples, 72, num_channels)
print("Combined input shape:", X_seq.shape)

# Process the multiclass target "event_type"
le = LabelEncoder()
y_multiclass_int = le.fit_transform(df['event_type'])
y_multiclass = to_categorical(y_multiclass_int)  # shape: (n_samples, num_classes)

# Process the binary target "extreme"
y_extreme = df['extreme'].values  # shape: (n_samples,)



Combined input shape: (6918, 72, 29)


In [25]:
import pickle
with open('X_seq.pkl', 'wb') as f:
    pickle.dump(X_seq, f)
with open ('y_multiclass.pkl', 'wb') as f:
    pickle.dump(y_multiclass, f)
with open ('y_extreme.pkl', 'wb') as f:
    pickle.dump(y_extreme, f)


In [ ]:
df["extreme"].value_counts()

In [22]:
from sklearn.model_selection import train_test_split
import numpy as np

# Assuming y_multiclass is one-hot encoded, convert it to integer labels
y_multiclass_int = np.argmax(y_multiclass, axis=1)

# Combine multiclass and binary labels for stratification
stratify_labels = list(zip(y_multiclass_int, y_extreme))

# Split the dataset into training+validation and test sets (80% train+val, 20% test)
X_train_val, X_test, y_train_val_multi, y_test_multi, y_train_val_ext, y_test_ext = train_test_split(
    X_seq, y_multiclass, y_extreme, test_size=0.2, random_state=42, stratify=stratify_labels
)

# Update stratify_labels for the training+validation set
stratify_labels_train_val = list(zip(np.argmax(y_train_val_multi, axis=1), y_train_val_ext))

# Split the training+validation set into training and validation sets (75% train, 25% val)
X_train, X_val, y_train_multi, y_val_multi, y_train_ext, y_val_ext = train_test_split(
    X_train_val, y_train_val_multi, y_train_val_ext, test_size=0.25, random_state=42, stratify=stratify_labels_train_val
)

# Verify the shapes of the splits
print(f'Training set size: {X_train.shape[0]} samples')
print(f'Validation set size: {X_val.shape[0]} samples')
print(f'Test set size: {X_test.shape[0]} samples')


Training set size: 21081 samples
Validation set size: 7028 samples
Test set size: 7028 samples


## Trying bidirectional LSTM

In [36]:
# ===========================
# Step 2. Build the Hybrid Multi-Task RNN Model
# ===========================
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import AUC

from sklearn.metrics import confusion_matrix, f1_score
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
import numpy as np
import tensorflow as tf

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from keras.callbacks import Callback
import numpy as np

from sklearn.metrics import roc_auc_score, f1_score
import numpy as np
from tensorflow.keras.callbacks import Callback

from keras.callbacks import Callback
from sklearn.metrics import roc_auc_score, f1_score

class MetricsCallback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    def on_epoch_end(self, epoch, logs=None):
        val_data, val_multi_labels, val_labels = self.validation_data
        
        # Obtain predictions from the model
        predictions = self.model.predict(val_data)
        
        # If the model has multiple outputs, ensure you extract the correct one
        if isinstance(predictions, list) or isinstance(predictions, tuple):
            val_multi_pred, val_predictions = predictions
        else:
            val_predictions = predictions
        
        # Ensure val_predictions is a 1D array
        val_predictions = val_predictions.reshape(-1)
        
        # Binarize predictions based on a threshold (e.g., 0.5)
        val_pred_labels = (val_predictions > 0.5).astype(int)
        auc = roc_auc_score(val_labels, val_predictions)
        f1 = f1_score(val_labels, val_pred_labels)
        # Compute AUC and F1 score for the 'extreme' classification
        precision = precision_score(val_labels, val_pred_labels)
        recall = recall_score(val_labels, val_pred_labels)
        
        print(f' — val_auc: {auc:.4f} — val_f1: {f1:.4f} — val_precision: {precision:.4f} — val_recall: {recall:.4f}')




# Input shape is (72 time steps, num_channels)
sequence_length = X_seq.shape[1]  # 72
num_channels = X_seq.shape[2]     # number of prev_cols
num_classes = y_multiclass.shape[1]
def create_model():


    input_layer = Input(shape=(sequence_length, num_channels), name='input_sequence')
    # Bidirectional LSTM layers
    x = Bidirectional(LSTM(128, return_sequences=True))(input_layer)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(64))(x)
    x = Dropout(0.3)(x)

    # Output for multiclass classification
    output_multiclass = Dense(num_classes, activation='softmax', name='event_type')(x)

    # Output for extreme vs. non-extreme classification
    output_extreme = Dense(1, activation='sigmoid', name='extreme')(x)

    # Model definition
    model = Model(inputs=input_layer, outputs=[output_multiclass, output_extreme])

    # Optimizer with gradient clipping
    optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

    auc_metric = AUC(name='auc')

    # Compile the model with the AUC metric
    model.compile(optimizer=optimizer,
                loss={'event_type': 'categorical_crossentropy', 'extreme': 'binary_crossentropy'},
                metrics={'event_type': 'accuracy', 'extreme': [auc_metric, 'accuracy']})
    return model
# Define early stopping callback monitoring 'val_extreme_auc'
early_stopping = EarlyStopping(monitor='val_extreme_auc', patience=5, mode='max', restore_best_weights=True)
validation_data = (X_val, {'event_type': y_val_multi, 'extreme': y_val_ext})

metrics_callback = MetricsCallback(validation_data=validation_data)



NameError: name 'X_val' is not defined

In [70]:
num_classes

9

## Trying LSTM CNN

In [56]:
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (Reshape, Conv2D, MaxPooling2D, Bidirectional,
                                     LSTM, Dropout, Dense, Flatten, Activation,
                                     RepeatVector, Permute, Multiply, Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
import tensorflow.keras.backend as K



def create_cnn_lstm_model():
        # Input layer
    input_layer = Input(shape=(sequence_length, num_channels), name='input_sequence')

    # CNN feature extraction
    x = Reshape((sequence_length, num_channels, 1))(input_layer)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 1))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = Reshape((sequence_length//2, -1))(x)

    # Bidirectional LSTM with attention
    x = Bidirectional(LSTM(256, return_sequences=True))(x)
    x = Dropout(0.3)(x)

    # Attention mechanism
    attention = Dense(1, activation='tanh')(x)
    attention = Flatten()(attention)
    attention = Activation('softmax')(attention)
    attention = RepeatVector(512)(attention)  # 512 = 256*2 for bidirectional
    attention = Permute([2, 1])(attention)
    attended = Multiply()([x, attention])
    x = Lambda(lambda x: K.sum(x, axis=1))(attended)

    # Final dense layers
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    output_multiclass = Dense(num_classes, activation='softmax', name='event_type')(x)
    output_extreme = Dense(1, activation='sigmoid', name='extreme')(x)
    model = Model(inputs=input_layer, outputs=[output_multiclass, output_extreme])

    # Optimizer with gradient clipping
    optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

    auc_metric = AUC(name='auc')

    # Compile the model with the AUC metric
    model.compile(optimizer=optimizer,
                loss={'event_type': 'categorical_crossentropy', 'extreme': 'binary_crossentropy'},
                metrics={'event_type': 'accuracy', 'extreme': [auc_metric, 'accuracy']})
    return model

In [68]:
y_multiclass.shape

(6918, 9)

In [72]:
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np

# Initialize StratifiedKFold with desired parameters
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Assuming X, y_ext, and y_multi are your input and output arrays
X, y_ext, y_multi = X_seq, y_extreme, y_multiclass
y_combined = list(zip(y_multiclass, y_extreme))
y_combined = np.array(['{}_{}'.format(a, b) for a, b in y_combined])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Lists to store metrics for both outputs
auc_scores_event_type = []
f1_scores_event_type = []
auc_scores_extreme = []
f1_scores_extreme = []

# Loop through the cross-validation folds
for train_index, val_index in skf.split(X, y_extreme):
    # Split data into training and validation based on the current fold
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold_event_type = y_multi[train_index]
    y_val_fold_event_type = y_multi[val_index]
    y_train_fold_extreme = y_ext[train_index]
    y_val_fold_extreme = y_ext[val_index]
    
    # Initialize a new instance of the model (ensure you have the create_model() function)
    model = create_cnn_lstm_model()  # This function should return a compiled model
    
    # Train the model with early stopping and metrics callback
    history = model.fit(
        X_train_fold,
        {'event_type': y_train_fold_event_type, 'extreme': y_train_fold_extreme},
        validation_data=(X_val_fold, {'event_type': y_val_fold_event_type, 'extreme': y_val_fold_extreme}),
        epochs=30,
        batch_size=64,
        callbacks=[early_stopping, MetricsCallback(validation_data=(X_val_fold, y_val_fold_event_type, y_val_fold_extreme))]
    )

    # Evaluate the model on the validation fold
    val_predictions = model.predict(X_val_fold)
    
    # The output from the model can be a list of predictions (one for each label)
    # Assuming val_predictions[0] corresponds to the 'event_type' class output
    # and val_predictions[1] corresponds to the 'extreme' class output
    val_pred_labels_event_type = (val_predictions[0] > 0.5).astype(int)
    val_pred_labels_extreme = (val_predictions[1] > 0.5).astype(int)
    
    # # Compute AUC and F1 score for the event_type label
    # auc_event_type = roc_auc_score(y_val_fold_event_type, val_predictions[0])
    # f1_event_type = f1_score(y_val_fold_event_type, val_pred_labels_event_type)
    
    # # Compute AUC and F1 score for the extreme label
    # auc_extreme = roc_auc_score(y_val_fold_extreme, val_predictions[1])
    # f1_extreme = f1_score(y_val_fold_extreme, val_pred_labels_extreme)
    
    # # Append scores for this fold
    # auc_scores_event_type.append(auc_event_type)
    # f1_scores_event_type.append(f1_event_type)
    # auc_scores_extreme.append(auc_extreme)
    # f1_scores_extreme.append(f1_extreme)
    
    # # Print metrics for this fold
    # print(f'Fold AUC (event_type): {auc_event_type:.4f}, Fold F1 (event_type): {f1_event_type:.4f}')
    # print(f'Fold AUC (extreme): {auc_extreme:.4f}, Fold F1 (extreme): {f1_extreme:.4f}')

# Compute average metrics over all folds
# print(f'Mean AUC (event_type): {np.mean(auc_scores_event_type):.4f}, Mean F1 (event_type): {np.mean(f1_scores_event_type):.4f}')
# print(f'Mean AUC (extreme): {np.mean(auc_scores_extreme):.4f}, Mean F1 (extreme): {np.mean(f1_scores_extreme):.4f}')


Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/stepp - event_type_accuracy: 0.5391 - event_type_loss: 1.1618 - extreme_accuracy: 0.5316 - extreme_auc: 
 — val_auc: 0.6441 — val_f1: 0.3402 — val_precision: 0.6933 — val_recall: 0.2254
87/87 ━━━━━━━━━━━━━━━━━━━━ 52s 552ms/step - event_type_accuracy: 0.5398 - event_type_loss: 1.1599 - extreme_accuracy: 0.5319 - extreme_auc: 0.5485 - extreme_loss: 0.7051 - loss: 1.8649 - val_event_type_accuracy: 0.6445 - val_event_type_loss: 0.8846 - val_extreme_accuracy: 0.5629 - val_extreme_auc: 0.6445 - val_extreme_loss: 0.6865 - val_loss: 1.5695
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/stepp - event_type_accuracy: 0.6795 - event_type_loss: 0.8218 - extreme_accuracy: 0.5965 - extreme_auc: 
 — val_auc: 0.6649 — val_f1: 0.6429 — val_precision: 0.5917 — val_recall: 0.7038
87/87 ━━━━━━━━━━━━━━━━━━━━ 52s 601ms/step - event_type_accuracy: 0.6794 - event_type_loss: 0.8219 - extreme_accuracy: 0.5965 - extreme_auc: 0.6368 - extreme_loss: 0.6660 - loss: 1.4879

KeyboardInterrupt: 

## trying VIT

In [73]:
import numpy as np
from pyts.image import GramianAngularField
from tqdm import tqdm
# Assuming X_seq has shape (n_samples, 72, 29)
n_samples, n_timestamps, n_channels = X_seq.shape

# Initialize the GramianAngularField transformer
gaf_transformer = GramianAngularField(image_size=n_timestamps)

# Transform each channel independently and concatenate the results
X_images = np.concatenate([
    gaf_transformer.fit_transform(channel.reshape(-1, n_timestamps))[:, :, np.newaxis]
    for channel in tqdm(X_seq.transpose((2, 0, 1)))
], axis=-1)

print("Transformed input shape:", X_images.shape)


100%|██████████| 29/29 [00:22<00:00,  1.31it/s]


Transformed input shape: (6918, 72, 1, 2088)


In [99]:
import numpy as np



# Save the array to a .npy file without using pickle
np.save('array_file.npy', X_images, allow_pickle=False)


In [12]:
import numpy as np
X_images = np.load('array_file.npy')

In [19]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Layer to extract patches from the image
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

# Layer to encode patches with a linear projection and add positional embedding
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patches)
        encoded += self.position_embedding(positions)
        return encoded

# MLP block used both in the transformer block and the classifier head.
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

# Create the Vision Transformer classifier
def create_vit_classifier(
    image_size=72,      # height and width of the input image
    patch_size=16,      # size of each patch
    num_channels=3,     # number of image channels (e.g. 3 for RGB)
    num_classes=2,      # number of classification labels
    projection_dim=128, # patch embedding dimension
    transformer_layers=6,   # number of transformer blocks
    num_heads=8,            # number of attention heads
    transformer_units=[256],# MLP units inside transformer block (expansion)
    mlp_head_units=[256]    # MLP units for the classification head
):
    # Calculate the number of patches.
    num_patches = (image_size // patch_size) ** 2

    inputs = keras.Input(shape=(72, 1, 2088))
    # Create patches.
    patches = Patches(patch_size)(inputs)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Build multiple transformer blocks.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Multi-head attention.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # Apply MLP block.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Project MLP output back to projection_dim so it matches x2.
        x3 = layers.Dense(projection_dim)(x3)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Classification head.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.1)(representation)
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.1)
    logits = layers.Dense(num_classes)(features)

    model = keras.Model(inputs=inputs, outputs=logits)
    return model

# Example of creating and compiling the model:
if __name__ == '__main__':
    vit_model = create_vit_classifier(
        image_size=72,
        patch_size=16,
        num_channels=3,
        num_classes=2,
        projection_dim=64,
        transformer_layers=3,
        num_heads=2,
        transformer_units=[256],  # You could also try [256, 128] if you prefer
        mlp_head_units=[256]
    )
    
    vit_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    vit_model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 72, 1,     │          0 │ -                 │
│ (InputLayer)        │ 2088)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patches_5 (Patches) │ (None, None,      │          0 │ input_layer_5[0]… │
│                     │ 534528)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder_5     │ (None, 16, 64)    │ 34,210,880 │ patches_5[0][0]   │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 64)    │        128 │ patch_encoder_5[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 64)    │     33,216 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_34 (Add)        │ (None, 16, 64)    │          0 │ multi_head_atten… │
│                     │                   │            │ patch_encoder_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 64)    │        128 │ add_34[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 16, 256)   │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_41          │ (None, 16, 256)   │          0 │ dense_44[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 16, 64)    │     16,448 │ dropout_41[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_35 (Add)        │ (None, 16, 64)    │          0 │ dense_45[0][0],   │
│                     │                   │            │ add_34[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 64)    │        128 │ add_35[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 64)    │     33,216 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_36 (Add)        │ (None, 16, 64)    │          0 │ multi_head_atten… │
│                     │                   │            │ add_35[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 64)    │        128 │ add_36[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 16, 256)   │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_43          │ (None, 16, 256)   │          0 │ dense_46[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_47 (Dense)    │ (None, 16, 64)    │     16,448 │ dropout_43[0][0]

 Total params: 34,673,602 (132.27 MB)

 Trainable params: 34,673,602 (132.27 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = vit_model.fit(
    X_images[:500], y_extreme[:500],
    epochs=50,            # Adjust number of epochs as needed.
    batch_size=2,
    validation_split=0.2  # You can also provide a separate test set.
)

Epoch 1/50


ValueError: Exception encountered when calling PatchEncoder.call().

[1mDimensions must be equal, but are 0 and 16 for '{{node functional_3_1/patch_encoder_5_1/add}} = AddV2[T=DT_FLOAT](functional_3_1/patch_encoder_5_1/dense_43_1/BiasAdd, functional_3_1/patch_encoder_5_1/embedding_5_1/GatherV2)' with input shapes: [2,0,64], [16,64].[0m

Arguments received by PatchEncoder.call():
  • patches=tf.Tensor(shape=(2, 0, 534528), dtype=float32)

## trying notmal transformers


In [49]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder



# ===========================
# Transformer Model Definition
# ===========================
def create_transformer_model(input_shape, num_heads, head_size, ff_dim, dropout_rate, num_transformer_blocks, num_multiclass):
    inputs = Input(shape=input_shape)

    x = inputs
    # Create several transformer encoder blocks using pre-built Keras layers
    for _ in range(num_transformer_blocks):
        # MultiHeadAttention block (pre available transformer cell)
        attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout_rate)(x, x)
        attn_output = Dropout(dropout_rate)(attn_output)
        x = LayerNormalization(epsilon=1e-6)(x + attn_output)
        
        # Feed-forward network (two Dense layers with a residual connection)
        ffn = Dense(ff_dim, activation='relu')(x)
        ffn = Dropout(dropout_rate)(ffn)
        ffn = Dense(x.shape[-1])(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)
    
    # Pool over the time dimension
    x = GlobalAveragePooling1D()(x)
    
    # Create two output heads:
    # 1. Binary output for extreme weather prediction.
    output_extreme = Dense(1, activation='sigmoid', name='extreme')(x)
    # 2. Multiclass output for event type classification.
    output_multiclass = Dense(num_multiclass, activation='softmax', name='event_type')(x)
    
    model = Model(inputs=inputs, outputs=[output_extreme, output_multiclass])
    return model

# Model hyperparameters (adjust as needed)
input_shape = X_seq.shape[1:]  # (72, num_channels)
num_heads = 4
head_size = 64
ff_dim = 128
dropout_rate = 0.1
num_transformer_blocks = 6
num_multiclass = y_multiclass.shape[1]

# Create and compile the model
model = create_transformer_model(input_shape, num_heads, head_size, ff_dim, dropout_rate, num_transformer_blocks, num_multiclass)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss={'extreme': 'binary_crossentropy', 'event_type': 'categorical_crossentropy'},
    metrics={'extreme': 'accuracy', 'event_type': 'accuracy'}
)

model.summary()

# ===========================
# Train the Model
# ===========================
# Split the dataset into train and test sets
X_train, X_test, y_train_extreme, y_test_extreme, y_train_multiclass, y_test_multiclass = train_test_split(
    X_seq,
    y_extreme,
    y_multiclass,
    test_size=0.2,
    random_state=42
)

validation_data = (X_test, {'extreme': y_test_extreme, 'event_type': y_test_multiclass})


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 72, 29)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 72, 29)    │     30,493 │ input_layer_13[0… │
│ (MultiHeadAttentio… │                   │            │ input_layer_13[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_91          │ (None, 72, 29)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_68 (Add)        │ (None, 72, 29)    │          0 │ input_layer_13[0… │
│                     │                   │            │ dropout_91[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 72, 29)    │         58 │ add_68[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_80 (Dense)    │ (None, 72, 128)   │      3,840 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_92          │ (None, 72, 128)   │          0 │ dense_80[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_81 (Dense)    │ (None, 72, 29)    │      3,741 │ dropout_92[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_69 (Add)        │ (None, 72, 29)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_81[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 72, 29)    │         58 │ add_69[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 72, 29)    │     30,493 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_94          │ (None, 72, 29)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_70 (Add)        │ (None, 72, 29)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_94[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 72, 29)    │         58 │ add_70[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_82 (Dense)    │ (None, 72, 128)   │      3,840 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_95          │ (None, 72, 128)   │          0 │ dense_82[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_83 (Dense)    │ (None, 72, 29)    │      3,741 │ dropout_95[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_71 (Add)        │ (None, 72, 29)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_83[0][0]  

 Total params: 229,440 (896.25 KB)

 Trainable params: 229,440 (896.25 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

class MetricsCallback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    def on_epoch_end(self, epoch, logs=None):
        val_data, val_labels_dict = self.validation_data
        val_labels_extreme = val_labels_dict['extreme']
        val_labels_event_type = val_labels_dict['event_type']
        
        # Obtain predictions from the model
        predictions = self.model.predict(val_data)
        
        # Extract predictions for each output
        val_predictions_extreme, val_predictions_event_type = predictions
        
        # Ensure val_predictions_extreme is a 1D array
        val_predictions_extreme = val_predictions_extreme.reshape(-1)
        
        # Binarize predictions based on a threshold (e.g., 0.5)
        val_pred_labels_extreme = (val_predictions_extreme > 0.5).astype(int)
        
        # Compute metrics for the 'extreme' classification
        auc_extreme = roc_auc_score(val_labels_extreme, val_predictions_extreme)
        f1_extreme = f1_score(val_labels_extreme, val_pred_labels_extreme)
        precision_extreme = precision_score(val_labels_extreme, val_pred_labels_extreme)
        recall_extreme = recall_score(val_labels_extreme, val_pred_labels_extreme)
        
        print(f' — val_auc_extreme: {auc_extreme:.4f} — val_f1_extreme: {f1_extreme:.4f} — val_precision_extreme: {precision_extreme:.4f} — val_recall_extreme: {recall_extreme:.4f}')
        
        # Compute metrics for the 'event_type' classification
        val_pred_labels_event_type = np.argmax(val_predictions_event_type, axis=1)
        val_true_labels_event_type = np.argmax(val_labels_event_type, axis=1)
        
        f1_event_type = f1_score(val_true_labels_event_type, val_pred_labels_event_type, average='weighted')
        precision_event_type = precision_score(val_true_labels_event_type, val_pred_labels_event_type, average='weighted')
        recall_event_type = recall_score(val_true_labels_event_type, val_pred_labels_event_type, average='weighted')
        
        print(f' — val_f1_event_type: {f1_event_type:.4f} — val_precision_event_type: {precision_event_type:.4f} — val_recall_event_type: {recall_event_type:.4f}')

In [51]:
# Combine the target variables into dictionaries for training
y_train_dict = {'extreme': y_train_extreme, 'event_type': y_train_multiclass}
y_test_dict = {'extreme': y_test_extreme, 'event_type': y_test_multiclass}
history = model.fit(
    X_train, 
    y_train_dict, 
    validation_data=(X_test, y_test_dict), 
    epochs=10, 
    batch_size=32,
    callbacks=[MetricsCallback(validation_data=validation_data)]
)


Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/steptep - event_type_accuracy: 0.5082 - event_type_loss: 1.3149 - extreme_accuracy: 0.5290 - extreme_l
 — val_auc_extreme: 0.6398 — val_f1_extreme: 0.6399 — val_precision_extreme: 0.5950 — val_recall_extreme: 0.6923
 — val_f1_event_type: 0.5768 — val_precision_event_type: 0.5851 — val_recall_event_type: 0.6149
173/173 ━━━━━━━━━━━━━━━━━━━━ 103s 163ms/step - event_type_accuracy: 0.5087 - event_type_loss: 1.3134 - extreme_accuracy: 0.5292 - extreme_loss: 0.7208 - loss: 2.0342 - val_event_type_accuracy: 0.6149 - val_event_type_loss: 0.9312 - val_extreme_accuracy: 0.5975 - val_extreme_loss: 0.6622 - val_loss: 1.5984
Epoch 2/10
  1/173 ━━━━━━━━━━━━━━━━━━━━ 28s 168ms/step - event_type_accuracy: 0.5938 - event_type_loss: 1.0519 - extreme_accuracy: 0.6562 - extreme_loss: 0.6396 - loss: 1.6915

c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/steptep - event_type_accuracy: 0.6295 - event_type_loss: 0.8976 - extreme_accuracy: 0.5894 - ext
 — val_auc_extreme: 0.6659 — val_f1_extreme: 0.5640 — val_precision_extreme: 0.6705 — val_recall_extreme: 0.4867
 — val_f1_event_type: 0.6001 — val_precision_event_type: 0.5991 — val_recall_event_type: 0.6221
173/173 ━━━━━━━━━━━━━━━━━━━━ 28s 164ms/step - event_type_accuracy: 0.6295 - event_type_loss: 0.8976 - extreme_accuracy: 0.5894 - extreme_loss: 0.6685 - loss: 1.5661 - val_event_type_accuracy: 0.6221 - val_event_type_loss: 0.9026 - val_extreme_accuracy: 0.6113 - val_extreme_loss: 0.6571 - val_loss: 1.5659
Epoch 3/10
  1/173 ━━━━━━━━━━━━━━━━━━━━ 30s 179ms/step - event_type_accuracy: 0.9062 - event_type_loss: 0.5435 - extreme_accuracy: 0.5312 - extreme_loss: 0.6893 - loss: 1.2328

c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/steptep - event_type_accuracy: 0.6432 - event_type_loss: 0.8680 - extreme_accuracy: 0.6039 - extreme_loss
 — val_auc_extreme: 0.6850 — val_f1_extreme: 0.6718 — val_precision_extreme: 0.6253 — val_recall_extreme: 0.7259
 — val_f1_event_type: 0.6004 — val_precision_event_type: 0.6012 — val_recall_event_type: 0.6257
173/173 ━━━━━━━━━━━━━━━━━━━━ 28s 160ms/step - event_type_accuracy: 0.6431 - event_type_loss: 0.8682 - extreme_accuracy: 0.6039 - extreme_loss: 0.6644 - loss: 1.5325 - val_event_type_accuracy: 0.6257 - val_event_type_loss: 0.8834 - val_extreme_accuracy: 0.6337 - val_extreme_loss: 0.6390 - val_loss: 1.5301
Epoch 4/10


...

c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/steptep - event_type_accuracy: 0.6767 - event_type_loss: 0.7996 - extreme_accuracy: 0.6444 - extreme_loss
 — val_auc_extreme: 0.7146 — val_f1_extreme: 0.6737 — val_precision_extreme: 0.6406 — val_recall_extreme: 0.7105
 — val_f1_event_type: 0.6279 — val_precision_event_type: 0.6542 — val_recall_event_type: 0.6496
173/173 ━━━━━━━━━━━━━━━━━━━━ 23s 131ms/step - event_type_accuracy: 0.6767 - event_type_loss: 0.7997 - extreme_accuracy: 0.6444 - extreme_loss: 0.6230 - loss: 1.4227 - val_event_type_accuracy: 0.6496 - val_event_type_loss: 0.8483 - val_extreme_accuracy: 0.6445 - val_extreme_loss: 0.6162 - val_loss: 1.4710
Epoch 10/10
  1/173 ━━━━━━━━━━━━━━━━━━━━ 25s 147ms/step - event_type_accuracy: 0.7812 - event_type_loss: 0.6840 - extreme_accuracy: 0.6250 - extreme_loss: 0.6557 - loss: 1.3397

c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/steptep - event_type_accuracy: 0.6898 - event_type_loss: 0.7908 - extreme_accuracy: 0.6585 - extreme_los
 — val_auc_extreme: 0.7170 — val_f1_extreme: 0.6606 — val_precision_extreme: 0.6624 — val_recall_extreme: 0.6587
 — val_f1_event_type: 0.6352 — val_precision_event_type: 0.6402 — val_recall_event_type: 0.6532
173/173 ━━━━━━━━━━━━━━━━━━━━ 24s 137ms/step - event_type_accuracy: 0.6897 - event_type_loss: 0.7909 - extreme_accuracy: 0.6584 - extreme_loss: 0.6199 - loss: 1.4108 - val_event_type_accuracy: 0.6532 - val_event_type_loss: 0.8352 - val_extreme_accuracy: 0.6503 - val_extreme_loss: 0.6120 - val_loss: 1.4538


c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
